In [1]:
2

2

## Imports

In [2]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import sqlite3
from datetime import datetime
from typing import List, Any
import json
import time

e:\AI course\assignment_answer\assignment 1\AI-Personal-Finance-Tracker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Openai API Key

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing . Please set it in .env file.")

client = OpenAI(api_key=openai_api_key)
MODEL = "gpt-4o" 

## DB Setup

In [4]:
DB_NAME = "finance.db"

In [5]:
def init_db():
    """Create tables if they don't exist."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Salary table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS salary (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            amount REAL NOT NULL,
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Expenses table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            amount REAL NOT NULL,
            category TEXT NOT NULL,
            description TEXT,
            date TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Savings goals table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS savings_goals (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            target_amount REAL NOT NULL,
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    conn.commit()
    conn.close()

init_db()

## Tool Functions

In [6]:
def set_salary(amount: float, month: str, year: int) -> str:

    print("calling set salary tool")
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    #if salary record found for same month and year, delete it
    cursor.execute("DELETE FROM salary WHERE month = ? AND year = ?", (month, year))
    cursor.execute(
        "INSERT INTO salary(amount, month, year) VALUES (?,?,?)",
        (amount, month, year)
    )
    conn.commit()
    conn.close()
    return f"Salary of ${amount:.2f} for {month} {year} is saved."


In [7]:
def log_expense(amount: float, category: str, description: str, date: str | None = None) -> str:

    print(f"[DEBUG] log_expense called: amount={amount}, category={category}, description={description}, date={date}")

    if not date:
        date = datetime.now().strftime("%Y-%m-%d")

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO expenses (amount, category, description, date) VALUES (?,?,?,?)",
        (amount, category, description, date)
    )
    conn.commit()

    expense_dt = datetime.strptime(date, "%Y-%m-%d")
    month_name = expense_dt.strftime("%B")
    year_val   = expense_dt.year
    month_str  = expense_dt.strftime("%Y-%m")

##Aditional feature to get the temp_remaining after log a expense##
# 1. to get the temp remaining, first fetch the salary
    cursor.execute(
        "SELECT amount FROM salary WHERE month=? AND year=? ORDER BY id DESC LIMIT 1",
        (month_name, year_val)
    )
    row    = cursor.fetchone()
    salary = row[0] if row else 0.0
# calculate total spend that month
    cursor.execute(
        "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
        (month_str,)
    )
    total_spent = cursor.fetchone()[0]
    conn.close()
# retun log expense + temp_total
    temp_remaining = salary - total_spent
    return (
        f"Logged the expesne of ${amount:.2f} for {category} ({description}) on {date}. "
        f"{month_name} {year_val} — Spent: ${total_spent:.2f} | Remaining: ${temp_remaining:.2f}"
    )


In [8]:
def get_balance(month: str | None = None, year: int | None = None) -> str:
    now = datetime.now()
    t_month = month if month else now.strftime("%B")
    t_year  = year  if year  else now.year

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

#fetch salary
    cursor.execute(
        "SELECT amount FROM salary WHERE month=? AND year=? ORDER BY id DESC LIMIT 1",
        (t_month, t_year)
    )
    row    = cursor.fetchone()
    salary = row[0] if row else 0.0

# calculate total spend this month
    month_str = datetime.strptime(f"{t_month} {t_year}", "%B %Y").strftime("%Y-%m")
    cursor.execute(
        "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
        (month_str,)
    )
    total_spent = cursor.fetchone()[0]
    conn.close()

    if salary == 0:
        return f"No salary record found for {t_month} {t_year}. Tell me your income first!"

# calculate remaining
    remaining = salary - total_spent
    return (
        f"{t_month} {t_year} Balance:\n"
        f"  Salary:    ${salary:.2f}\n"
        f"  Spent:     ${total_spent:.2f}\n"
        f"  Remaining: ${remaining:.2f}"
    )


In [9]:
def get_expense_summary(month: str | None = None, year: int | None = None) -> str:
    now = datetime.now()
    t_month = month if month else now.strftime("%B")
    t_year  = year  if year  else now.year

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT amount FROM salary WHERE month=? AND year=? ORDER BY id DESC LIMIT 1",
        (t_month, t_year)
    )
    row    = cursor.fetchone()
    salary = row[0] if row else 0.0

    month_str = datetime.strptime(f"{t_month} {t_year}", "%B %Y").strftime("%Y-%m")
    cursor.execute(
        "SELECT category, SUM(amount) FROM expenses WHERE strftime('%Y-%m', date) = ? GROUP BY category ORDER BY SUM(amount) DESC",
        (month_str,)
    )
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No expenses logged for {t_month} {t_year} yet."

    total_spent = sum(r[1] for r in rows)
    lines = [f"{t_month} {t_year} Spending Breakdown:"]
    #expenses as a percentage
    for cat, amt in rows:
        pct = (amt / salary * 100) if salary > 0 else 0
        lines.append(f"  - {cat:<14} ${amt:.2f}  ({pct:.1f}% of salary)")

    lines.append(f"Total spent for the month is: ${total_spent:.2f} | Remaining: ${salary - total_spent:.2f}")
    return "\n".join(lines)


## Additional Tools 

In [10]:
def set_savings_goal(amount: float, month: str, year: int) -> str:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM savings_goals WHERE month = ? AND year = ?", (month, year))
    cursor.execute(
        "INSERT INTO savings_goals (target_amount, month, year) VALUES (?, ?, ?)",
        (amount, month, year)
    )
    conn.commit()
    conn.close()
    return f"Savings goal of ${amount:.2f} set for {month} {year}."


In [11]:
def check_savings_progress(month: str | None = None, year: int | None = None) -> str:
    now = datetime.now()
    t_month = month if month else now.strftime("%B")
    t_year  = year  if year  else now.year

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
#fetch salary
    cursor.execute(
        "SELECT amount FROM salary WHERE month=? AND year=? ORDER BY id DESC LIMIT 1",
        (t_month, t_year)
    )
    row    = cursor.fetchone()
    salary = row[0] if row else 0.0

#get the all expenses relevant to the target month
    month_str = datetime.strptime(f"{t_month} {t_year}", "%B %Y").strftime("%Y-%m")
    cursor.execute(
        "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
        (month_str,)
    )
    total_spent = cursor.fetchone()[0]

    cursor.execute(
        "SELECT target_amount FROM savings_goals WHERE month=? AND year=? ORDER BY id DESC LIMIT 1",
        (t_month, t_year)
    )
    goal_row = cursor.fetchone()
    conn.close()

    actual_saved = salary - total_spent

    if goal_row is None:
        return (
            f"No savings goal set for {t_month} {t_year}. "
            f"Current savings: ${actual_saved:.2f}. "
            f"Set a goal, e.g. 'I want to save $500 in {t_month}'."
        )

    goal = goal_row[0]
    if salary == 0:
        return f"No salary set for {t_month} {t_year}."

    if actual_saved >= goal:
        return (
            f"{t_month} {t_year} Savings Progress:\n"
            f"  Goal:     ${goal:.2f}\n"
            f"  Saved:    ${actual_saved:.2f}\n"
            f"  Status:   Goal reached! ${actual_saved - goal:.2f} ahead."
        )

    shortfall = goal - actual_saved
    return (
        f"{t_month} {t_year} Savings Progress:\n"
        f"  Goal:      ${goal:.2f}\n"
        f"  Saved:     ${actual_saved:.2f}\n"
        f"  Shortfall: ${shortfall:.2f} still needed to achieve your saving target."
    )

In [12]:
def delete_last_expense() -> str:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id, amount, category, description, date FROM expenses ORDER BY id DESC LIMIT 1"
    )
    row = cursor.fetchone()
    if not row:
        conn.close()
        return "No expenses found to delete."
    exp_id, amount, category, description, date = row
    cursor.execute("DELETE FROM expenses WHERE id = ?", (exp_id,))
    conn.commit()
    conn.close()
    return f"Deleted: ${amount:.2f} for {category} ({description}) on {date}."


In [13]:
def compare_months(month1: str, year1: int, month2: str, year2: int) -> str:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    #call for both months
    data = {}
    for m, y in [(month1, year1), (month2, year2)]:
        #Converts "April 2026" → "2026-04" for filtering expenses
        month_str = datetime.strptime(f"{m} {y}", "%B %Y").strftime("%Y-%m")
        #get salary
        cursor.execute(
            "SELECT amount FROM salary WHERE month=? AND year=? ORDER BY id DESC LIMIT 1", (m, y)
        )
         #Gets spending by category like a dictionary
        row    = cursor.fetchone()
        salary = row[0] if row else 0.0
        cursor.execute(
            "SELECT category, SUM(amount) FROM expenses "
            "WHERE strftime('%Y-%m', date) = ? GROUP BY category",
            (month_str,)
        )
        cats  = dict(cursor.fetchall())
        #get all total monthly expenses
        cursor.execute(
            "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
            (month_str,)
        )
        total = cursor.fetchone()[0]

        #store collected data
        data[(m, y)] = {"salary": salary, "cats": cats, "total": total}

    conn.close()

    r1, r2   = data[(month1, year1)], data[(month2, year2)]
    all_cats = sorted(set(r1["cats"]) | set(r2["cats"]))
    lbl1, lbl2 = f"{month1[:3]} {year1}", f"{month2[:3]} {year2}"
    sep = "-" * 56
    #display
    lines = [f"Comparison: {month1} {year1} vs {month2} {year2}", sep]
    lines.append(f"  {'':<16}  {lbl1:>10}  {lbl2:>10}  {'Change':>10}")
    lines.append(sep)
    lines.append(f"  {'Salary':<16}  ${r1['salary']:>9.2f}  ${r2['salary']:>9.2f}")
    lines.append(sep)

    for cat in all_cats:
        a1    = r1["cats"].get(cat, 0.0)
        a2    = r2["cats"].get(cat, 0.0)
        delta = a2 - a1
        sign  = "+" if delta > 0 else ("-" if delta < 0 else " ")
        lines.append(f"  {cat:<16}  ${a1:>9.2f}  ${a2:>9.2f}  {sign}${abs(delta):.2f}")

    lines.append(sep)
    dt   = r2["total"] - r1["total"]
    sign = "+" if dt > 0 else ("-" if dt < 0 else " ")
    lines.append(f"  {'Total Spent':<16}  ${r1['total']:>9.2f}  ${r2['total']:>9.2f}  {sign}${abs(dt):.2f}")
    lines.append(f"  {'Saved':<16}  ${r1['salary']-r1['total']:>9.2f}  ${r2['salary']-r2['total']:>9.2f}")

    return "\n".join(lines)

In [14]:
tools: List[Any] = [
    {
        "type": "function",
        "function": {
            "name": "set_salary",
            "description": "Save the user's monthly salary or income. Call this when the user mentions their salary, income, or earnings for a specific month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Salary amount"},
                    "month": {"type": "string", "description": "Month name, e.g. April"},
                    "year": {"type": "integer", "description": "Year, e.g. 2026"}
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_expense",
            "description": "Record a new expense. Use this whenever the user mentions spending money on anything — rent, groceries, coffee, subscriptions, etc. Infer category from context (Rent, Groceries, Dining, Transport, Entertainment, Healthcare, Shopping). If the user mentions a past month (e.g. 'in January'), set date to a day in that month like YYYY-01-15.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Expense amount"},
                    "category": {"type": "string", "description": "Expense category, e.g. Rent, Groceries, Dining"},
                    "description": {"type": "string", "description": "Short description of what was purchased"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format. Use today if no date given. If a past month is mentioned, use the 15th of that month."}
                },
                "required": ["amount", "category", "description"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get salary, total spent, and remaining balance for a given month. If no month is specified, use the current month. Use when the user asks how much is left, what their balance is, or how they are tracking.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {"type": "string", "description": "Month name, e.g. January. Omit to use current month."},
                    "year": {"type": "integer", "description": "Year, e.g. 2026. Omit to use current year."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": "Show expenses grouped by category for a given month. If no month is specified, use the current month. Use when the user asks for a breakdown, summary, or wants to know where their money went.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {"type": "string", "description": "Month name, e.g. January. Omit to use current month."},
                    "year": {"type": "integer", "description": "Year, e.g. 2026. Omit to use current year."}
                },
                "required": []
            }
        }
    }
,
    {
        "type": "function",
        "function": {
            "name": "set_savings_goal",
            "description": "Set a monthly savings target. Call this when the user says they want to save a specific amount in a month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number",  "description": "Target savings amount"},
                    "month":  {"type": "string",  "description": "Month name, e.g. April"},
                    "year":   {"type": "integer", "description": "Year, e.g. 2026"}
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_savings_progress",
            "description": "Check how the user is tracking against their savings goal. Use when the user asks if they are on track or how much they have saved.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {"type": "string",  "description": "Month name, e.g. April. Omit for current month."},
                    "year":  {"type": "integer", "description": "Year, e.g. 2026. Omit for current year."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_last_expense",
            "description": "Delete the most recently logged expense. Use when the user says they made a mistake, logged a wrong amount, or wants to undo the last entry.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare spending side by side between two months, showing each category and the change. Use when the user asks to compare two months or see how spending changed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month1": {"type": "string",  "description": "First month name, e.g. March"},
                    "year1":  {"type": "integer", "description": "First month year, e.g. 2026"},
                    "month2": {"type": "string",  "description": "Second month name, e.g. April"},
                    "year2":  {"type": "integer", "description": "Second month year, e.g. 2026"}
                },
                "required": ["month1", "year1", "month2", "year2"]
            }
        }
    }
]

## Describe Tools to AI

In [15]:
def run_agent(user_message: str, history: List[Any]) -> str:
    """
    Core agent loop:
    1. Append user message to conversation
    2. Call OpenAI with tools
    3. If tool calls, execute them, append results, and loop again
    4. Return final text response
    """
    now = datetime.now()
    system_prompt = (
        f"You are a friendly, usefull, intelligent personal finance assistant. "
        f"Today is {now.strftime('%A, %B %d, %Y')}. Current month is {now.strftime('%B')} {now.year}. "
        "Help users track their income and expenses through natural conversation. "
        "When the user mentione salary without specifying a month, assume the current month. "
        "When the user asks for a summary or balance of a past month (e.g. 'January' or 'last month'), "
        "pass that month and year to the tool. "
        "Always be concise and friendly in your responses."
    )

    messages: List[Any] = [{"role": "system", "content": system_prompt}]
    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": user_message})

    available_functions = {
        "set_salary": set_salary,
        "log_expense": log_expense,
        "get_balance": get_balance,
        "get_expense_summary":    get_expense_summary,
        "set_savings_goal":       set_savings_goal,
        "check_savings_progress": check_savings_progress,
        "delete_last_expense":    delete_last_expense,
        "compare_months":         compare_months,
    }

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3,
        )

        response_message = response.choices[0].message

        if response_message.tool_calls:
            messages.append(response_message.model_dump())

            for tool_call in response_message.tool_calls:
                if not hasattr(tool_call, "function"):
                    continue
                function_name = tool_call.function.name  # type: ignore
                function_args = json.loads(tool_call.function.arguments)  # type: ignore

                print(f"[DEBUG] calling tool: {function_name} with args: {function_args}")

                if function_name in available_functions:
                    result = available_functions[function_name](**function_args)
                else:
                    result = f"Error: tool '{function_name}' not found."

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue
        else:
            return response_message.content or ""

# UI implementation

In [ ]:

def chat_fn(message, history):
    """Wrapper for Gradio ChatInterface with typewriter streaming effect."""
    formatted_history = []
    if history:
        if isinstance(history[0], dict):
            i = 0
            while i < len(history):
                if history[i].get("role") == "user":
                    user_msg = history[i]["content"]
                    asst_msg = history[i+1]["content"] if i+1 < len(history) and history[i+1].get("role") == "assistant" else ""
                    formatted_history.append((user_msg, asst_msg))
                    i += 2
                else:
                    i += 1
        else:
            formatted_history = [(h[0], h[1]) for h in history]

    response = run_agent(message, formatted_history)

    partial = ""
    for char in response:
        partial += char
        yield partial
        time.sleep(0.012)

# Custom CSS
custom_css = """
    .gradio-container {
        max-width: 1200px !important;
        margin: auto !important;
    }

    /* Extra bottom space so the fixed input bar never covers messages */
    body { padding-bottom: 120px !important; }

    .header-section {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 2rem;
        border-radius: 15px;
        margin-bottom: 1.5rem;
        color: white;
    }

    .feature-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
        gap: 1rem;
        margin: 0rem 0;
    }

    .feature-card {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        padding: 1rem;
        border-radius: 10px;
        text-align: center;
        transition: transform 0.3s ease;
    }

    .feature-card:hover { transform: translateY(-5px); }

    .message { border-radius: 15px !important; margin: 0.5rem 0 !important; }
    .user    { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; }
    .bot     { background: #f0f2f5 !important; color: #1a1a1a !important; border-left: 4px solid #667eea !important; }

    .chat-input       { border-radius: 25px !important; border: 2px solid #e0e0e0 !important; transition: all 0.3s ease !important; }
    .chat-input:focus { border-color: #667eea !important; box-shadow: 0 0 0 2px rgba(102,126,234,0.1) !important; }

    .example-button       { background: linear-gradient(135deg, #f5f7fa 0%, #e8eef5 100%) !important; border: 1px solid #d0d7de !important; border-radius: 20px !important; padding: 0.75rem 1rem !important; margin: 0.25rem !important; transition: all 0.2s ease !important; }
    .example-button:hover { background: linear-gradient(135deg, #e8eef5 0%, #dce3ec 100%) !important; transform: translateY(-2px); }

    h1, h2, h3 { font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; }

    @media (max-width: 768px) {
        .feature-grid   { grid-template-columns: 1fr; }
        .header-section { padding: 1rem; }
    }
"""
# the viewport — exactly like ChatGPT.  Gradio renders asynchronously so we
FIXED_INPUT_JS = """
<script>
(function () {
    function findInputRow() {
        var textarea = document.querySelector('.gradio-container textarea');
        if (!textarea) return null;

        // both the textarea AND a submit button (that is the input row).
        var el = textarea.parentElement;
        for (var i = 0; i < 10; i++) {
            if (!el || el.classList.contains('gradio-container')) break;
            if (el.querySelector('button') && el.querySelector('textarea') && el.children.length >= 2) {
                return el;
            }
            el = el.parentElement;
        }
        return null;
    }

    function applyFixed(row) {
        row.style.setProperty('position',   'fixed',                               'important');
        row.style.setProperty('bottom',     '0',                                   'important');
        row.style.setProperty('left',       '0',                                   'important');
        row.style.setProperty('right',      '0',                                   'important');
        row.style.setProperty('z-index',    '9999',                                'important');
        row.style.setProperty('background', '#ffffff',                             'important');
        row.style.setProperty('padding',    '0.75rem 2rem 1.25rem',                'important');
        row.style.setProperty('box-shadow', '0 -4px 24px rgba(102,126,234,0.18)', 'important');
        row.style.setProperty('border-top', '2px solid #e8eef5',                   'important');
    }

    var done = false, tries = 0;
    function attempt() {
        if (done || tries++ > 30) return;
        var row = findInputRow();
        if (row) { applyFixed(row); done = true; return; }
        setTimeout(attempt, 500);
    }

    window.addEventListener('load', function () { setTimeout(attempt, 800); });
    setTimeout(attempt, 1500);
    setTimeout(attempt, 3500);   // final safety-net retry
})();
</script>
"""

with gr.Blocks(
    title="Personal Finance Tracker",
    theme=gr.themes.Soft(  # type: ignore
        primary_hue="indigo",
        secondary_hue="purple",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Inter")  # type: ignore
    ),
    css=custom_css
) as demo:

    # Header
    with gr.Row(elem_classes=["header-section"]):
        with gr.Column(scale=2):
            gr.Markdown("""
            # Personal Finance Assistant
            ### Your AI-powered financial companion
            """)
        with gr.Column(scale=2):
            gr.Markdown("""
            <p style="font-size:1.1rem;opacity:0.95;">
            Talk naturally like you're chatting with a friend. I'll help you track every dollar
            and achieve your financial goals with ease.
            </p>
            """)

    # Feature grid
    with gr.Row():
        with gr.Column():
            gr.HTML("""
            <div class="feature-grid">
                <div class="feature-card">📊 <strong>Track Income</strong><br><small>Record salary & earnings</small></div>
                <div class="feature-card">💸 <strong>Monitor Expenses</strong><br><small>Log daily spending</small></div>
                <div class="feature-card">📈 <strong>Budget Analysis</strong><br><small>Get spending breakdowns</small></div>
                <div class="feature-card">🎯 <strong>Smart Insights</strong><br><small>AI-powered recommendations</small></div>
            </div>
            """)

    # Chat (full width)
    with gr.Row():
        with gr.Column():
            gr.ChatInterface(
                fn=chat_fn,
                examples=[
                    "My monthly salary is $2000",
                    "I spend $100 for my monthly rent",
                    "Spend 50$ for nails and hair",
                    "How much do I have left?",
                    "Show me my spending breakdown",
                    "What's my biggest expense this month?",
                    "Help me save more money",
                    "I want to save $500 this month",
                    "Am I on track with my savings goal?",
                    "Oops I logged the wrong expense, delete it",
                    "Compare March and April spending",
                ],
            )

    # Pro Tips
    with gr.Row():
        with gr.Column():
            gr.Markdown("""
            ### 💡 Pro Tips
            -  **Be specific about amounts** — e.g. *"Paid $45 for groceries"*
            -  **Mention dates for accuracy** — e.g. *"Last Tuesday I spent $20 on lunch"*
            -  **Categorize your spending** — helps you spot patterns faster
            - **Set monthly saving goals** — ask me to track your progress
            """)


    # Inject the fixed-input script — must live inside the Blocks context
    gr.HTML(FIXED_INPUT_JS)

if __name__ == "__main__":
    demo.launch(share=False, debug=False, show_error=True, quiet=False)


C:\Users\User\AppData\Local\Temp\ipykernel_19824\672817868.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# For testing purpose - to clean up db tables

In [17]:
def delete_all_expenses():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("DELETE FROM expenses")
    conn.commit()
    conn.close()
    print("All records in expenses tabel are deleted.")

def delete_all_salary():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("DELETE FROM salary")
    conn.commit()
    conn.close()
    print("All records in salary table are deleted.")

def delete_all_savings_goals():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("DELETE FROM savings_goals")
    conn.commit()
    conn.close()
    print("All records in savings_goals table are deleted.")

def show_all():
    conn = sqlite3.connect(DB_NAME)
    expenses      = conn.execute("SELECT * FROM expenses").fetchall()
    salary        = conn.execute("SELECT * FROM salary").fetchall()
    savings_goals = conn.execute("SELECT * FROM savings_goals").fetchall()
    conn.close()
    print(f"Expenses      ({len(expenses)} rows):",      expenses)
    print(f"Salary        ({len(salary)} rows):",        salary)
    print(f"Savings Goals ({len(savings_goals)} rows):", savings_goals)


In [18]:
#delete_all_expenses()

In [19]:
#delete_all_salary() 

In [20]:
#delete_all_savings_goals()

In [21]:
show_all() 

Expenses      (3 rows): [(49, 100.0, 'Rent', 'Rent', '2026-04-29', '2026-04-29 17:23:08'), (50, 20.0, 'Shopping', 'Saloon', '2026-04-29', '2026-04-29 17:23:09'), (51, 50.0, 'Groceries', 'Groceries', '2026-04-29', '2026-04-29 17:23:09')]
Salary        (2 rows): [(25, 2000.0, 'April', 2026, '2026-04-29 17:22:40'), (26, 1500.0, 'March', 2026, '2026-04-29 17:26:41')]
Savings Goals (1 rows): [(4, 500.0, 'April', 2026, '2026-04-29 17:24:58')]
